In [ ]:
%load_ext autoreload
%reload_ext autoreload

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from generate_data import UserGenerator

In [9]:
generator_data = UserGenerator(seed=42, n_samples=10000)
df = generator_data.create_dataset()

df.head()

Generando datos sintéticos de usuarios para targeting de promociones...


,user_id,age_group,location,device_type,subscription_type,days_since_registration,total_purchases,avg_order_value,last_purchase_days,sessions_last_30_days,time_on_site_minutes,pages_per_session,cart_abandonment_rate,purchase_frequency,dar_promocion
0,USER-000001,18-25,Mendoza,Mobile,Free,38,6,341.58,139,2,71.3,1.6,0.075,4.74,1
1,USER-000002,26-35,Buenos Aires,Mobile,Basic,246,34,215.56,114,18,34.1,17.5,0.607,4.15,0
2,USER-000003,36-45,Buenos Aires,Mobile,Enterprise,284,21,60.08,97,3,43.7,7.5,0.212,2.22,1
3,USER-000004,36-45,Cordoba,Tablet,Free,343,35,153.66,160,19,106.4,7.9,0.154,3.06,0
4,USER-000005,18-25,Buenos Aires,Mobile,Free,330,14,434.58,97,8,55.0,16.8,0.130,1.27,1


## 1. Análisis Exploratorio de Datos (EDA)

**¿Qué es EDA?** Es el proceso de investigar y entender nuestros datos antes de construir modelos de machine learning. Es como conocer a tu equipo antes del partido.

**¿Por qué es importante?**
- 🎯 **Entender las variables**: Qué significan y cómo se distribuyen
- 🔍 **Detectar problemas**: Valores faltantes, outliers, errores en los datos
- 📊 **Identificar patrones**: Relaciones entre variables que puedan ser útiles
- 💡 **Generar ideas**: Para crear nuevas características o mejorar el modelo

En esta sección vamos a:</cell>
</invoke>

In [ ]:
# Histograma del valor promedio de orden con Plotly
fig = px.histogram(df, x='avg_order_value', 
                   title='Distribución del Valor Promedio de Orden',
                   labels={'avg_order_value': 'Valor Promedio de Orden ($)', 'count': 'Frecuencia'},
                   nbins=30,
                   color_discrete_sequence=['#1f77b4'])

fig.update_layout(
    showlegend=False, 
    height=400,
    xaxis_title="Valor Promedio de Orden ($)",
    yaxis_title="Frecuencia"
)
fig.show()

# Estadísticas descriptivas
print(f"📊 Estadísticas del Valor Promedio de Orden:")
print(f"  Media: ${df['avg_order_value'].mean():.2f}")
print(f"  Mediana: ${df['avg_order_value'].median():.2f}")
print(f"  Desv. Estándar: ${df['avg_order_value'].std():.2f}")
print(f"  Mínimo: ${df['avg_order_value'].min():.2f}")
print(f"  Máximo: ${df['avg_order_value'].max():.2f}")

# Distribución de la variable objetivo
target_counts = df['dar_promocion'].value_counts()
fig2 = px.bar(x=target_counts.index, 
              y=target_counts.values,
              title='Distribución de la Variable Objetivo (dar_promocion)',
              labels={'x': 'Dar Promoción', 'y': 'Cantidad'},
              color=target_counts.index,
              color_discrete_map={0: 'lightcoral', 1: 'lightgreen'})

fig2.update_layout(height=300, showlegend=False)
fig2.show()

print(f"📊 Balance de clases:")
print(f"  No promoción (0): {(df['dar_promocion'] == 0).sum()} ({(df['dar_promocion'] == 0).mean():.1%})")
print(f"  Dar promoción (1): {(df['dar_promocion'] == 1).sum()} ({(df['dar_promocion'] == 1).mean():.1%})")

In [ ]:
# Visualizaciones adicionales para entender mejor los datos

# 1. Matriz de correlación interactiva
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

fig_corr = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.round(2).values,
    texttemplate="%{text}",
    textfont={"size": 10},
    hoverongaps=False
))

fig_corr.update_layout(
    title='Matriz de Correlación de Variables Numéricas',
    height=600,
    width=700
)
fig_corr.show()

# 2. Distribuciones por variable categórica
fig_cat = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Grupo de Edad', 'Ubicación', 'Tipo de Dispositivo', 'Tipo de Suscripción'],
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "bar"}]]
)

categorical_cols = ['age_group', 'location', 'device_type', 'subscription_type']
colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']

for i, col in enumerate(categorical_cols):
    row = i // 2 + 1
    col_num = i % 2 + 1
    
    counts = df[col].value_counts()
    
    fig_cat.add_trace(
        go.Bar(x=counts.index, y=counts.values, 
               marker_color=colors[i], name=col,
               showlegend=False),
        row=row, col=col_num
    )

fig_cat.update_layout(height=500, title_text="Distribución de Variables Categóricas")
fig_cat.show()

# 3. Boxplots de variables numéricas importantes por variable objetivo
important_numeric = ['avg_order_value', 'total_purchases', 'sessions_last_30_days', 'purchase_frequency']

fig_box = make_subplots(
    rows=2, cols=2,
    subplot_titles=[col.replace('_', ' ').title() for col in important_numeric]
)

for i, col in enumerate(important_numeric):
    row = i // 2 + 1
    col_num = i % 2 + 1
    
    # Datos para no promoción
    fig_box.add_trace(
        go.Box(y=df[df['dar_promocion'] == 0][col], 
               name='No Promoción', marker_color='red',
               showlegend=(i==0)),
        row=row, col=col_num
    )
    
    # Datos para promoción
    fig_box.add_trace(
        go.Box(y=df[df['dar_promocion'] == 1][col], 
               name='Dar Promoción', marker_color='green',
               showlegend=(i==0)),
        row=row, col=col_num
    )

fig_box.update_layout(height=500, title_text="Distribuciones por Variable Objetivo")
fig_box.show()

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  10000 non-null  object 
 1   age_group                10000 non-null  object 
 2   location                 10000 non-null  object 
 3   device_type              10000 non-null  object 
 4   subscription_type        10000 non-null  object 
 5   days_since_registration  10000 non-null  int64  
 6   total_purchases          10000 non-null  int64  
 7   avg_order_value          10000 non-null  float64
 8   last_purchase_days       10000 non-null  int64  
 9   sessions_last_30_days    10000 non-null  int64  
 10  time_on_site_minutes     10000 non-null  float64
 11  pages_per_session        10000 non-null  float64
 12  cart_abandonment_rate    10000 non-null  float64
 13  purchase_frequency       10000 non-null  float64
 14  dar_promocion          

## 2. Ingeniería de Características (Feature Engineering)

**¿Qué es Feature Engineering?** Es el arte de crear nuevas variables a partir de las existentes para ayudar al modelo a entender mejor los patrones.

**¿Por qué es crucial?** 
- 🧠 **Mejora la performance**: Nuevas características pueden revelar patrones ocultos
- 📈 **Dominio del negocio**: Incorporamos conocimiento específico del problema
- 🔧 **Transformaciones útiles**: Ratios, diferencias, agrupaciones que tienen sentido

**Estrategias comunes:**
- **Ratios**: Dividir una variable por otra (ej: compras por día)
- **Diferencias**: Restar variables relacionadas (ej: tiempo entre eventos)
- **Binning**: Agrupar valores continuos en categorías
- **Interacciones**: Combinar múltiples variables

In [ ]:
# Crear nuevas características (feature engineering)

# 1. Compras por día (para medir actividad)
df['total_purchases_per_day'] = df['total_purchases'] / df['days_since_registration']

# 2. Días entre primera y última compra (para medir engagement temporal)
df["days_between_first_and_last_purchase"] = df["days_since_registration"] - df["last_purchase_days"]

# 3. Crear buckets para el valor promedio de orden usando quantiles
# pd.qcut es más robusto que pd.cut para este caso
try:
    df["bucket_avg_order_value"] = pd.qcut(
        df["avg_order_value"], 
        q=3, 
        labels=["low", "medium", "high"],
        duplicates='drop'  # Maneja valores duplicados automáticamente
    )
    print("✅ Buckets creados exitosamente usando quantiles")
except Exception as e:
    print(f"⚠️ Error: {e}")
    # Alternativa: usar valores fijos si quantiles fallan
    df["bucket_avg_order_value"] = pd.cut(
        df["avg_order_value"], 
        bins=[-np.inf, 100, 300, np.inf], 
        labels=["low", "medium", "high"]
    )
    print("✅ Buckets creados usando rangos fijos")

# Mostrar las nuevas características
print("\n📊 Nuevas características creadas:")
print(f"  total_purchases_per_day - Media: {df['total_purchases_per_day'].mean():.4f}")
print(f"  days_between_first_and_last_purchase - Media: {df['days_between_first_and_last_purchase'].mean():.1f}")
print(f"  bucket_avg_order_value - Distribución:")
print(df['bucket_avg_order_value'].value_counts())

## 3. División de Datos (Train/Test Split)

**¿Por qué dividir los datos?** 
- 🎯 **Evaluar honestamente**: El modelo nunca debe ver los datos de prueba durante entrenamiento
- 🚫 **Evitar overfitting**: Prevenir que el modelo memorice en lugar de aprender patrones generales
- 📊 **Simular producción**: Los datos de prueba representan datos futuros no vistos

**Consideraciones importantes:**
- **Proporción típica**: 70-80% entrenamiento, 20-30% prueba
- **Estratificación**: Mantener la misma proporción de clases en ambos conjuntos
- **Aleatoriedad controlada**: Usar random_state para reproducibilidad

In [14]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=["dar_promocion"]), df["dar_promocion"], test_size=0.2, random_state=42)

## 4. Pipeline de Procesamiento y Entrenamiento

**¿Qué es un Pipeline?** Es una secuencia automatizada de pasos de procesamiento y modelado. Es como una línea de ensamblaje para machine learning.

**Ventajas de usar Pipelines:**
- 🔄 **Reproducibilidad**: Los mismos pasos se aplican consistentemente
- 🚀 **Eficiencia**: Automatizan el flujo de trabajo completo
- 🛡️ **Prevención de errores**: Evitan data leakage y errores de procesamiento
- 🔧 **Mantenimiento**: Fácil de modificar y actualizar

**Componentes del Pipeline:**
1. **Preprocesamiento**: Limpieza, transformación, escalado
2. **Modelo**: Algoritmo de machine learning
3. **Predicción**: Aplicación automática de todos los pasos

### 4.1 Procesamiento de Variables Numéricas

**Decisiones de preprocesamiento para variables numéricas:**

**¿Por qué imputar?** Los algoritmos de ML no pueden trabajar con valores faltantes.
- 📊 **Mediana**: Robusta a outliers, buena opción general
- 📈 **Media**: Sensible a outliers pero preserva la distribución
- 🔢 **Cero**: Cuando los valores faltantes tienen significado específico
- 📋 **Moda**: Para variables casi categóricas

**¿Por qué escalar?** Los algoritmos basados en distancia (como regresión logística) son sensibles a la escala.
- ⚖️ **StandardScaler**: Centra en 0 con desviación 1 (distribución normal)
- 📏 **MinMaxScaler**: Escala entre 0 y 1 (bueno para algoritmos que necesitan valores positivos)
- 🔄 **RobustScaler**: Menos sensible a outliers

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

# Define numeric and categorical columns
numeric_features = ['days_since_registration', 'total_purchases', 'avg_order_value', 
                   'last_purchase_days', 'sessions_last_30_days', 'time_on_site_minutes', 
                   'pages_per_session', 'cart_abandonment_rate', 'purchase_frequency']

# Create preprocessing pipelines for each data type
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

### 4.2 Procesamiento de Variables Categóricas

**Decisiones de preprocesamiento para variables categóricas:**

**¿Por qué imputar con "Unknown"?** 
- 🏷️ **Información explícita**: "Unknown" es más claro que eliminar filas
- 📊 **Preserva datos**: No perdemos observaciones por valores faltantes
- 🎯 **Patrón potencial**: A veces los valores faltantes son informativos

**¿Por qué OneHotEncoder?**
- 🔢 **Convierte texto a números**: Los algoritmos necesitan valores numéricos
- ⚖️ **No asume orden**: A diferencia de LabelEncoder, no implica que "A" < "B"
- 🎛️ **drop='first'**: Evita multicolinealidad perfecta
- 🛡️ **handle_unknown='ignore'**: Maneja categorías nuevas en producción

In [19]:
categorical_features = ['age_group', 'location', 'device_type', 'subscription_type']

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

### 4.3 ColumnTransformer: Uniendo Todo

**¿Qué hace ColumnTransformer?**
- 🔗 **Unifica transformaciones**: Aplica diferentes procesamientos a diferentes tipos de columnas
- 🎯 **Selectividad**: Especifica exactamente qué columnas transformar
- ⚡ **Eficiencia**: Aplica transformaciones en paralelo
- 🛡️ **Consistencia**: Garantiza que el mismo procesamiento se use en entrenamiento y predicción

**Parámetros importantes:**
- `remainder='drop'`: Elimina columnas no especificadas (como user_id)
- `transformers`: Lista de (nombre, transformador, columnas)

In [20]:
# Combine transformers using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'  # Drop any columns not specified above
)

### 4.4 Pipeline Final: Preprocesamiento + Modelo

**¿Por qué Regresión Logística?**
- 📊 **Problema de clasificación**: Queremos predecir si dar promoción o no (binario)
- 🧠 **Interpretable**: Podemos entender qué variables son importantes
- ⚡ **Rápida**: Entrena rápidamente incluso con muchas características
- 🎯 **Probabilidades**: Nos da probabilidades, no solo predicciones

**Parámetros del modelo:**
- `random_state=42`: Para reproducibilidad
- `max_iter=1000`: Suficientes iteraciones para convergencia

In [21]:
# Create the full pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

In [22]:
# Fit the pipeline
pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 5. Predicciones del Modelo

**Tipos de predicciones:**
- 🎯 **predict()**: Devuelve la clase predicha (0 o 1)
- 📊 **predict_proba()**: Devuelve probabilidades [P(clase=0), P(clase=1)]

**¿Por qué necesitamos probabilidades?**
- 🎚️ **Umbral ajustable**: Podemos cambiar el punto de corte (default 0.5)
- 📈 **Métricas avanzadas**: Para calcular AUC-ROC y otras métricas
- 💼 **Decisiones de negocio**: Diferentes costos para falsos positivos vs falsos negativos

In [23]:

# Make predictions
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

## 6. Evaluación del Modelo

**¿Cómo interpretamos los resultados?**

**ROC-AUC Score (0.515):**
- 📊 **Rango**: 0.5 (aleatorio) a 1.0 (perfecto)
- ❌ **Resultado**: ~0.51 significa que el modelo es apenas mejor que adivinar
- 🎯 **Objetivo**: Queremos valores > 0.7 para considerar el modelo útil

**Classification Report:**
- **Precision**: De los que predecimos como "dar promoción", ¿cuántos realmente la merecían?
- **Recall**: De todos los que merecían promoción, ¿cuántos identificamos?
- **F1-Score**: Media armónica de precision y recall
- **Accuracy**: Porcentaje de predicciones correctas totales

**🚨 Conclusión actual**: El modelo no está funcionando bien. Necesitamos mejorarlo.

In [26]:
# Evaluate
from sklearn.metrics import classification_report, roc_auc_score
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_proba))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

ROC-AUC Score: 0.5148348553827932

Classification Report:
              precision    recall  f1-score   support

           0       0.52      0.51      0.51      1019
           1       0.50      0.52      0.51       981

    accuracy                           0.51      2000
   macro avg       0.51      0.51      0.51      2000
weighted avg       0.51      0.51      0.51      2000



In [ ]:
# Visualizaciones de los resultados del modelo

# 1. Matriz de confusión
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

fig_cm = go.Figure(data=go.Heatmap(
    z=cm,
    x=['Pred: No Promoción', 'Pred: Dar Promoción'],
    y=['Real: No Promoción', 'Real: Dar Promoción'],
    colorscale='Blues',
    text=cm,
    texttemplate="%{text}",
    textfont={"size": 16}
))

fig_cm.update_layout(
    title='Matriz de Confusión',
    height=400,
    width=500
)
fig_cm.show()

# 2. Curva ROC
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

fig_roc = go.Figure()

# Curva ROC
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f'ROC Curve (AUC = {roc_auc_score(y_test, y_pred_proba):.3f})',
    line=dict(color='blue', width=2)
))

# Línea diagonal (random classifier)
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(color='red', dash='dash')
))

fig_roc.update_layout(
    title='Curva ROC',
    xaxis_title='Tasa de Falsos Positivos',
    yaxis_title='Tasa de Verdaderos Positivos',
    height=400,
    width=500
)
fig_roc.show()

# 3. Distribución de probabilidades predichas
fig_prob = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Distribución de Probabilidades', 'Por Clase Real']
)

# Histograma general
fig_prob.add_trace(
    go.Histogram(x=y_pred_proba, nbinsx=30, name='Probabilidades',
                marker_color='lightblue', opacity=0.7),
    row=1, col=1
)

# Histogramas por clase
fig_prob.add_trace(
    go.Histogram(x=y_pred_proba[y_test == 0], nbinsx=30, name='No Promoción',
                marker_color='red', opacity=0.6),
    row=1, col=2
)
fig_prob.add_trace(
    go.Histogram(x=y_pred_proba[y_test == 1], nbinsx=30, name='Dar Promoción',
                marker_color='green', opacity=0.6),
    row=1, col=2
)

fig_prob.update_layout(
    title='Análisis de Probabilidades Predichas',
    height=400
)
fig_prob.show()

print("\\n📊 Análisis de la matriz de confusión:")
tn, fp, fn, tp = cm.ravel()
print(f"  ✅ Verdaderos Negativos: {tn}")
print(f"  ❌ Falsos Positivos: {fp}")  
print(f"  ❌ Falsos Negativos: {fn}")
print(f"  ✅ Verdaderos Positivos: {tp}")
print(f"\\n  📈 Especificidad: {tn/(tn+fp):.3f}")
print(f"  📈 Sensibilidad (Recall): {tp/(tp+fn):.3f}")

## **Actividad**

Prueba:

1. Imputar los numericos con otras estrategias
2. Crea nuevas features
3. Usar BinaryEncoder para las categoricas
4. Usar MinMaxScaler para las numericas
5. Usar RandomForestClassifier para el modelo
6. Usar GridSearchCV para encontrar los mejores hiperparametros
7. Juega con el split de datos


Luego de hacer todas estas pruebas en el mismo notebook, crea un mensaje "resumen" de tus cambios y tus resultados. 

1+1